In [1]:
import os
os.chdir('/home/smallyan/eval_agent')
print(f"Working directory: {os.getcwd()}")

Working directory: /home/smallyan/eval_agent


# Generalizability Evaluation for Function Vectors Repository

This notebook evaluates whether the findings in the repository generalize beyond the original experimental setting.

## Evaluation Checklist:
- **GT1**: Generalization to a New Model
- **GT2**: Generalization to New Data  
- **GT3**: Method / Specificity Generalizability

## Repository Path
`/net/scratch2/smallyan/function_vectors_eval`

In [2]:
# First, let's explore the repository structure
import os

repo_path = '/net/scratch2/smallyan/function_vectors_eval'

# List all files and directories
for root, dirs, files in os.walk(repo_path):
    level = root.replace(repo_path, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 2 * (level + 1)
    for file in files[:20]:  # Limit files shown per directory
        print(f'{subindent}{file}')
    if len(files) > 20:
        print(f'{subindent}... and {len(files) - 20} more files')

function_vectors_eval/
  .gitignore
  fv_overview.png
  documentation.pdf
  plan.md
  CodeWalkthrough.md
  fv_environment.yml
  src/
    portability_eval.py
    test_numheads.py
    compute_indirect_effect.py
    vocab_reconstruction.py
    __init__.py
    compute_avg_hidden_state.py
    natural_text_eval.py
    evaluate_function_vector.py
    compute_average_activations.py
    __pycache__/
      __init__.cpython-311.pyc
      compute_indirect_effect.cpython-311.pyc
    utils/
      eval_utils.py
      prompt_utils.py
      intervention_utils.py
      extract_utils.py
      __init__.py
      model_utils.py
      __pycache__/
        model_utils.cpython-311.pyc
        intervention_utils.cpython-311.pyc
        __init__.cpython-311.pyc
        prompt_utils.cpython-311.pyc
        extract_utils.cpython-311.pyc
        eval_utils.cpython-311.pyc
    eval_scripts/
      eval_fv.sh
      eval_numheads.sh
      eval_template_portability.sh
      eval_avg_hs.sh
      template.sh
      fv_eval

      b8/
        4133e40737af5901448083ff12f2fbc60fd5e1
        6d084bae9c4c5f9b52f209e1e7a0fe95fc0e35
      1a/
        673feefa664c246ba46f9a88f6b8700dfe6172
        1debbd3e4feb9dd44969a573ea123784f1147a
        6258bfa0268e5b0bae93067dde998e144da013
        ce54eb300a7b1c7f02f06b2ab87884f74c49dc
      5d/
        0644398c85a7f8636bc63594ea1c652be0e56e
      a0/
        4e44866ce7a0a9951e0afc805309051b2baefe
      75/
        f197c11453ec94dc7923e35f74b6654cd37833
        99d4ad3431c2f63c02e78049d1d89e5a1892b7
        5e7cc2ca5de4617ffbdc129ec1dcc350c746ec
      2d/
        46ec792061db563402b2eda040698f6f0cbfb6
      05/
        1d5b41a55813c4dc55dbc885a112e2e666f4c7
      8e/
        729ace7148b677f4bc1925bbd6a07b52af50e8
      be/
        2f6a88600728c950a17b1d4877320b5ffec95f
      53/
        5e66d2c795e86b148afac369b240ffbe5b5ae0
      9b/
        4cbaaf6c5d8ec76e853eedd6dbd248e633820f
      c2/
        c650706b88c73110a63743ceb44ac3270dd08b
      3c/
        4ed044c4b8837a22

In [3]:
# Read the plan and documentation to understand the research
with open(os.path.join(repo_path, 'plan.md'), 'r') as f:
    plan_content = f.read()
print("=== PLAN.MD ===")
print(plan_content[:5000])

=== PLAN.MD ===
# Plan
## Objective
To investigate whether autoregressive transformer language models develop compact vector representations of input-output functions (called function vectors) within their hidden states during in-context learning, and to characterize these representations across diverse tasks and models.

## Hypothesis
1. A small number of attention heads transport a compact representation of the demonstrated task (function vector) that is robust to changes in context and can trigger task execution in zero-shot and natural text settings.
2. Function vectors contain information encoding the output space of the function, but this information alone is not sufficient to reconstruct a working function vector.
3. Function vectors can be composed through vector algebra to create vectors that trigger new complex tasks combining constituent tasks.

## Methodology
1. Apply causal mediation analysis to identify attention heads with highest average indirect effect (AIE) across div

In [4]:
# Read the CodeWalkthrough to understand the implementation
with open(os.path.join(repo_path, 'CodeWalkthrough.md'), 'r') as f:
    walkthrough_content = f.read()
print("=== CODE WALKTHROUGH ===")
print(walkthrough_content[:8000])

=== CODE WALKTHROUGH ===
# Function Vectors in Large Language Models
### [Project Website](https://functions.baulab.info) | [Arxiv Preprint](https://arxiv.org/abs/2310.15213) | [OpenReview](https://openreview.net/forum?id=AwyxtyMwaG)

This repository contains data and code for the paper: [Function Vectors in Large Language Models](https://arxiv.org/pdf/2310.15213).

<p align="left">
<img src="https://functions.baulab.info/images/Paper/fv-demonstrations.png" style="width:100%;"/>
</p> 

## Setup

We recommend using conda as a package manager. 
The environment used for this project can be found in the `fv_environment.yml` file.
To install, you can run: 
```
conda env create -f fv_environment.yml
conda activate fv
```

## Demo Notebook
Checkout `notebooks/fv_demo.ipynb` for a jupyter notebook with a demo of how to create a function vector and use it in different contexts.

## Data
The datasets used in our project can be found in the `dataset_files` folder.

## Code
Our main evaluation scr

In [5]:
# Read the demo notebook to understand how function vectors work
with open(os.path.join(repo_path, 'notebooks/fv_demo.ipynb'), 'r') as f:
    import json
    demo_notebook = json.load(f)

# Print the cells
for i, cell in enumerate(demo_notebook['cells'][:15]):
    print(f"=== Cell {i} ({cell['cell_type']}) ===")
    source = ''.join(cell['source'])
    print(source[:1500])
    print("\n")

=== Cell 0 (code) ===
%load_ext autoreload
%autoreload 2


=== Cell 1 (code) ===
import os, re, json
import torch, numpy as np

import sys
sys.path.append('..')
torch.set_grad_enabled(False)

from src.utils.extract_utils import get_mean_head_activations, compute_universal_function_vector
from src.utils.intervention_utils import fv_intervention_natural_text, function_vector_intervention
from src.utils.model_utils import load_gpt_model_and_tokenizer
from src.utils.prompt_utils import load_dataset, word_pairs_to_prompt_data, create_prompt
from src.utils.eval_utils import decode_to_vocab, sentence_eval


=== Cell 2 (markdown) ===
## Load model & tokenizer


=== Cell 3 (code) ===
model_name = 'EleutherAI/gpt-j-6b'
model, tokenizer, model_config = load_gpt_model_and_tokenizer(model_name)
EDIT_LAYER = 9


=== Cell 4 (markdown) ===
## Load dataset and Compute task-conditioned mean activations


=== Cell 5 (code) ===
dataset = load_dataset('antonym', seed=0)
mean_activations = get_mean_head_act

In [6]:
# Check existing evaluation files
eval_path = os.path.join(repo_path, 'evaluation')
print("Files in evaluation directory:")
for f in os.listdir(eval_path):
    print(f"  {f}")

Files in evaluation directory:
  consistency_evaluation.json
  self_matching.ipynb


In [7]:
# Read the consistency evaluation to understand what was evaluated
with open(os.path.join(eval_path, 'consistency_evaluation.json'), 'r') as f:
    consistency_eval = json.load(f)
print(json.dumps(consistency_eval, indent=2))

{
  "Checklist": {
    "CS1_Results_vs_Conclusion": "PASS",
    "CS2_Plan_vs_Implementation": "FAIL"
  },
  "Rationale": {
    "CS1_Results_vs_Conclusion": "All evaluable conclusions in the documentation are consistent with the code implementation. The methodology is correctly implemented in compute_indirect_effect.py, extract_utils.py, evaluate_function_vector.py, portability_eval.py, natural_text_eval.py, and vocab_reconstruction.py. The pre-computed top_heads in extract_utils.py match the documentation's description of using 10-100 heads based on model size. No contradictions were found between the documentation and the implementation.",
    "CS2_Plan_vs_Implementation": "Plan Step 4 (Vector Algebra Composition) is missing from the implementation. While Steps 1-3 are fully implemented, there is no dedicated script for inter-task FV composition (Eq. 7-8 from the paper). The experiments described in Table 7 (Last-Antonym, Last-Capitalize, Last-Country-Capital, etc.) involving composed

In [8]:
# Let's check the source utils to understand what models were used in the original research
with open(os.path.join(repo_path, 'src/utils/model_utils.py'), 'r') as f:
    model_utils_content = f.read()
print(model_utils_content)

import torch
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer, LlamaTokenizer, LlamaForCausalLM
import os
import random
from typing import *


def load_gpt_model_and_tokenizer(model_name:str, device='cuda', revision=None):
    """
    Loads a huggingface model and its tokenizer

    Parameters:
    model_name: huggingface name of the model to load (e.g. GPTJ: "EleutherAI/gpt-j-6B", or "EleutherAI/gpt-j-6b")
    device: 'cuda' or 'cpu'
    
    Returns:
    model: huggingface model
    tokenizer: huggingface tokenizer
    MODEL_CONFIG: config variables w/ standardized names
    
    """
    assert model_name is not None

    print("Loading: ", model_name)

    if model_name == 'gpt2-xl':
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        tokenizer.pad_token = tokenizer.eos_token
        model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

        MODEL_CONFIG={"n_heads":model.config.n_head,
                      "n_layers

In [9]:
# Look at extract_utils to understand how function vectors are computed
with open(os.path.join(repo_path, 'src/utils/extract_utils.py'), 'r') as f:
    extract_utils_content = f.read()
print(extract_utils_content[:5000])

import os, re, json

import torch, numpy as np
import pandas as pd
from baukit import TraceDict

# Include prompt creation helper functions
from .prompt_utils import *
from .intervention_utils import *
from .model_utils import *
from .eval_utils import *


# Attention Activations
def gather_attn_activations(prompt_data, layers, dummy_labels, model, tokenizer, model_config):
    """
    Collects activations for an ICL prompt 

    Parameters:
    prompt_data: dict containing ICL prompt examples, and template information
    layers: layer names to get activatons from
    dummy_labels: labels and indices for a baseline prompt with the same number of example pairs
    model: huggingface model
    tokenizer: huggingface tokenizer

    Returns:
    td: tracedict with stored activations
    idx_map: map of token indices to respective averaged token indices
    idx_avg: dict containing token indices of multi-token words
    """   
    
    # Get sentence and token labels
    query = prompt_dat

In [10]:
# Continue reading extract_utils
print(extract_utils_content[5000:10000])

 stack_initial = torch.vstack([split_activations_by_head(activations_td[layer].input, model_config) for layer in model_config['attn_hook_names']]).permute(0,2,1,3)
        stack_filtered = stack_initial[:,:,list(idx_map.keys())]
        for (i,j) in idx_avg.values():
            stack_filtered[:,:,idx_map[i]] = stack_initial[:,:,i:j+1].mean(axis=2) # Average activations of multi-token words across all its tokens
        
        activation_storage[n] = stack_filtered

    mean_activations = activation_storage.mean(dim=0)
    return mean_activations

# Layer Activations
def gather_layer_activations(prompt_data, layers, model, tokenizer, model_config):
    """
    Collects activations for an ICL prompt 

    Parameters:
    prompt_data: dict containing
    layers: layer names to get activatons from
    model: huggingface model
    tokenizer: huggingface tokenizer

    Returns:
    td: tracedict with stored activations
    """   
    
    # Get sentence and token labels
    query = prompt

In [11]:
# Continue reading extract_utils - looking for compute_universal_function_vector
print(extract_utils_content[10000:])

[i][1] for i in range(model_config['n_layers'])]) # (layers, heads, tokens, head_dim)
    value_weighted_attn = torch.einsum("abcd,abd->abcd", attentions, values.norm(dim=-1))
    return attentions, value_weighted_attn

def get_token_averaged_attention(dataset, model, model_config, tokenizer, n_shots=10, storage_max=100, filter_set=None):
    """

    Parameters:
    dataset: ICL dataset
    model: huggingface model
    model_config: dict with model information - n_layers, n_heads, etc.
    tokenizer: huggingface tokenizer
    n_shots: number of ICL example pairs to use for each prompt
    storage_max: max number of sentences to average attention pattern over
    filter_set: list of ints to filter to desired dataset instances

    Returns:
    attn_storage: attention heatmaps
    vw_attn_storage: value-weighted attention heatmaps
    token_labels: sample token labels for an n-shot prompt
    """
    if filter_set is not None:
        storage_size = min(len(filter_set), storage_max)
   

## Understanding the Repository

Based on the exploration, this repository implements **Function Vectors in Large Language Models** (ICLR 2024).

### Key Findings from the Paper:
1. **Function Vectors (FVs)**: Compact vector representations of input-output functions that emerge in transformer hidden states during in-context learning
2. **Top Causal Heads**: A small number of attention heads (10-100) transport the function vector information
3. **Pre-computed Universal Heads**: The code contains pre-computed top heads for:
   - GPT-J 6B
   - GPT-NeoX 20B  
   - Llama-2-7B, 13B, 70B

### Models Used in Original Work:
- GPT-J 6B
- GPT-NeoX 20B
- Llama 2 (7B, 13B, 70B)

### For Generalization Testing:
- **GT1 (New Model)**: We need to test on a model NOT in the original work. Options: Pythia, OLMo, Gemma (supported in model_utils.py)
- **GT2 (New Data)**: We need to test on data instances NOT in the original dataset
- **GT3 (Method)**: The method is causal mediation analysis to identify top heads + FV extraction. We can test if it applies to other similar tasks.

In [12]:
# Check GPU availability and setup environment
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

CUDA available: True
GPU: NVIDIA H200 NVL
GPU Memory: 150.1 GB


In [13]:
# Setup paths and imports
import sys
sys.path.insert(0, '/net/scratch2/smallyan/function_vectors_eval')

import os
import json
import numpy as np
import torch
torch.set_grad_enabled(False)

from src.utils.extract_utils import get_mean_head_activations, compute_universal_function_vector
from src.utils.intervention_utils import function_vector_intervention
from src.utils.model_utils import load_gpt_model_and_tokenizer, set_seed
from src.utils.prompt_utils import load_dataset, word_pairs_to_prompt_data, create_prompt
from src.utils.eval_utils import decode_to_vocab, sentence_eval

# Set seed for reproducibility
set_seed(42)
print("Imports successful!")

Imports successful!


In [14]:
# List available datasets
dataset_path = '/net/scratch2/smallyan/function_vectors_eval/dataset_files'
abstractive_datasets = os.listdir(os.path.join(dataset_path, 'abstractive'))
extractive_datasets = os.listdir(os.path.join(dataset_path, 'extractive'))

print("Abstractive datasets:")
for d in sorted(abstractive_datasets):
    print(f"  {d}")
    
print("\nExtractive datasets:")
for d in sorted(extractive_datasets):
    print(f"  {d}")

Abstractive datasets:
  ag_news.json
  antonym.json
  capitalize.json
  capitalize_first_letter.json
  capitalize_last_letter.json
  capitalize_second_letter.json
  commonsense_qa.json
  country-capital.json
  country-currency.json
  english-french.json
  english-german.json
  english-spanish.json
  landmark-country.json
  lowercase_first_letter.json
  lowercase_last_letter.json
  national_parks.json
  next_capital_letter.json
  next_item.json
  park-country.json
  person-instrument.json
  person-occupation.json
  person-sport.json
  present-past.json
  prev_item.json
  product-company.json
  sentiment.json
  singular-plural.json
  synonym.json
  word_length.json

Extractive datasets:
  adjective_v_verb_3.json
  adjective_v_verb_5.json
  alphabetically_first_3.json
  alphabetically_first_5.json
  alphabetically_last_3.json
  alphabetically_last_5.json
  animal_v_object_3.json
  animal_v_object_5.json
  choose_first_of_3.json
  choose_first_of_5.json
  choose_last_of_3.json
  choose_las

---
# GT1: Generalization to a New Model

We will test if the function vector methodology generalizes to a model NOT used in the original work.

**Models used in original work:** GPT-J 6B, GPT-NeoX 20B, Llama-2 (7B, 13B, 70B)

**New model for testing:** We will use **Pythia** (EleutherAI/pythia-1.4b or pythia-2.8b) which:
1. Is NOT in the original paper's model list
2. Is supported in the codebase (model_utils.py has pythia support)
3. Is small enough to run quickly for trial examples

**Test approach:**
1. Load the new model
2. Compute mean activations for a task (e.g., antonym)
3. Run causal mediation analysis to find top heads
4. Extract function vector using identified heads
5. Test if FV intervention improves zero-shot/shuffled performance

In [15]:
# GT1: Test on Pythia model (not used in original work)
# Using pythia-2.8b as it's a reasonable size

print("Loading Pythia-2.8b model...")
model_name = 'EleutherAI/pythia-2.8b'
model, tokenizer, model_config = load_gpt_model_and_tokenizer(model_name)
print(f"Model loaded successfully!")
print(f"Model config: {model_config['n_layers']} layers, {model_config['n_heads']} heads, {model_config['resid_dim']} dim")

Loading Pythia-2.8b model...
Loading:  EleutherAI/pythia-2.8b


`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/5.68G [00:00<?, ?B/s]

In [16]:
# Check if model is loaded
print(f"Model type: {type(model)}")
print(f"Model config: {model_config}")

In [17]:
# Wait a moment and check the model
import time
time.sleep(2)
print("Checking model status...")
try:
    print(f"Model loaded: {model is not None}")
    print(f"Model on device: {next(model.parameters()).device}")
except Exception as e:
    print(f"Error: {e}")

In [18]:
# Simple check
print("Cell executed")

In [19]:
print("Testing output")